# Submit Kaggle: RSNA Knee Abnormality Detection

Notebook nộp bài. Chỉ cần:
1. Bật **GPU** (Settings → Accelerator).
2. Thêm **dataset checkpoint** có file `student_best.pt` (upload thành Kaggle Dataset).
3. Dataset cuộc thi `rsna-knee-abnormality-detection` đã được gắn sẵn.

Notebook tự `git clone` repo, dựng predictor từ checkpoint và ghi `/kaggle/working/submission.csv`.
Chỉ dùng student (ảnh + metadata): không cần report, teacher, VLM hay SAM.

In [ ]:
# 0. Lấy code: git clone từ GitHub. Nếu không clone được (ví dụ khi Kaggle tắt Internet
#    lúc chấm bài), dùng bản code đã upload vào /kaggle/input.
import glob, subprocess, sys
from pathlib import Path

REPO_URL = 'https://github.com/hoangtung386/Att-kneenet.git'
CODE = Path('/kaggle/working/Att-kneenet')

if (CODE / '.git').is_dir():
    subprocess.run(['git', '-C', str(CODE), 'pull', '--ff-only'], check=False)
else:
    result = subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(CODE)], check=False)
    if result.returncode != 0:
        hits = glob.glob('/kaggle/input/**/src/knee_mri/__init__.py', recursive=True)
        if not hits:
            raise RuntimeError('Không clone được repo và cũng không thấy code trong /kaggle/input.')
        CODE = Path(hits[0]).parents[2]
        print('Dùng code từ /kaggle/input:', CODE)

if (CODE / 'pyproject.toml').is_file():
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(CODE)], check=False)
sys.path.insert(0, str(CODE / 'src'))
print('code:', CODE)

In [ ]:
# 1. Tìm dữ liệu cuộc thi và checkpoint trong /kaggle/input.
from pathlib import Path

def find_one(pattern):
    hits = sorted(glob.glob(pattern, recursive=True))
    return Path(hits[0]) if hits else None

TEST_CSV = find_one('/kaggle/input/competitions/rsna-knee-abnormality-detection/test.csv')
CHECKPOINT = find_one('/kaggle/input/models/hoangtung719/att-v1/pytorch/default/1/student_best.pt')

assert TEST_CSV is not None, 'Không thấy test.csv trong /kaggle/input — hãy gắn dataset cuộc thi.'
assert CHECKPOINT is not None, 'Không thấy student_best.pt — hãy thêm dataset checkpoint vào notebook.'
DATA_ROOT = TEST_CSV.parent
assert (DATA_ROOT / 'test_series.csv').is_file(), f'Thiếu test_series.csv trong {DATA_ROOT}'
assert (DATA_ROOT / 'test_series').is_dir(), f'Thiếu thư mục test_series trong {DATA_ROOT}'

print('dữ liệu :', DATA_ROOT)
print('checkpoint:', CHECKPOINT)

In [ ]:
# 2. Nạp cấu hình và chạy suy luận trên toàn bộ tập test.
import logging

import torch

from knee_mri.config import load_config
from knee_mri.inference.predictor import predict_submission

logging.basicConfig(level=logging.INFO)

# Phải dùng đúng cấu hình lúc train (volume 64×128×128); env 'local' dùng volume nhỏ 32×64×64 nên lệch số patch.
cfg = load_config('colab', overrides={
    'paths': {
        'data_root': str(DATA_ROOT),
        'artifacts_root': '/kaggle/working/artifacts',
        'dino_code_dir': '/kaggle/working/unused',   # không dùng khi suy luận
    },
})
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

OUTPUT = Path('/kaggle/working/submission.csv')
predict_submission(cfg, checkpoint=CHECKPOINT, output=OUTPUT, device=device)

In [ ]:
# 3. Kiểm tra file nộp: đủ số study, đúng cột với sample_submission.csv.
import pandas as pd

from knee_mri.constants import COL_STUDY, LABELS

submission = pd.read_csv(OUTPUT)
test = pd.read_csv(TEST_CSV)
print('shape:', submission.shape)
print(submission.head())

assert len(submission) == test[COL_STUDY].nunique(), 'Số dòng không khớp số study trong test.csv'
assert submission[list(LABELS)].notna().all().all(), 'Có xác suất bị thiếu (NaN)'

sample = DATA_ROOT / 'sample_submission.csv'
if sample.is_file():
    expected = list(pd.read_csv(sample, nrows=1).columns)
    assert list(submission.columns) == expected, f'Cột khác sample: {expected}'
    print('Cột khớp sample_submission.csv')
print('OK:', OUTPUT)